In [5]:
import re
import time
import shutil
import unicodedata
import numpy as np
import pandas as pd
import json

from os import path
from pathlib import Path
from datetime import datetime
from sqlalchemy import create_engine, text, types as sat

🧩 Bloque 1 – Importación de librerías

En este bloque se importan las librerías necesarias para:
- Manipular archivos Excel y datos con pandas y numpy.
- Conectarse a la base de datos SQL Server mediante SQLAlchemy.
- Trabajar con fechas, rutas y expresiones regulares.
    
Estas herramientas permiten la automatización del proceso completo de lectura, limpieza y carga de datos.

In [2]:
#from pathlib import Path
import sys
import importlib

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks
# queremos:  .../traspaso-innominados/functions
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

if module_name in sys.modules:
    fun = importlib.reload(sys.modules[module_name])
else:
    fun = importlib.import_module(module_name)

print("Importado desde:", getattr(fun, "__file__", "<sin __file__>"))

PROJECT_ROOT: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados
FUNCTIONS_DIR: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\functions exists: True
Importado desde: C:\Users\dasanchez\OneDrive - Seguros Suramericana, S.A\Documentos\data_analysis_projects\traspaso-innominados\functions\functions.py


🧩 Bloque 0 – Importación de configuraciones

In [7]:
path.join(PROJECT_ROOT, "config")

'C:\\Users\\dasanchez\\OneDrive - Seguros Suramericana, S.A\\Documentos\\data_analysis_projects\\traspaso-innominados\\config'

In [12]:
try:
    with open(path.join(PROJECT_ROOT, "config", "config_conosur.json"), 'r') as file:
        data = json.load(file)
    
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file.")

⚙️ Bloque 2 – Configuración de conexión a SQL Server

Aquí se definen los parámetros del servidor, base de datos, esquema y tabla destino.

Luego, se crea el motor de conexión con SQLAlchemy, que será utilizado para insertar los datos procesados en SQL Server.

La conexión usa autenticación de Windows y optimiza la velocidad con fast_executemany.

In [ ]:
# --- Parámetros iguales a los tuyos ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

# --- Nuevo: construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# Si tu entorno requiere TLS/cert (muy común con Driver 18), descomenta:
# connection_string += "Encrypt=yes;TrustServerCertificate=yes;"

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

Bloque 3 – Definición de ruta y hoja objetivo

Se indica la carpeta donde se encuentran los archivos Excel.

Se filtran los archivos válidos por extensión (.xlsx).

Se define la hoja preferida a leer (por nombre o índice).

Este bloque prepara el entorno para ubicar correctamente los archivos a procesar.

In [5]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "CONOSUR").resolve()


print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())


#RUTA_ARCHIVOS = Path(r"C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\CONOSUR")
EXTS = (".xlsx",)  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["Cesantia_Conosur", "cesantia_conosur", "conosur_cesantia"]
FALLBACK_SHEET_INDEX = 1

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
RUTA_ARCHIVOS: C:\Users\aepmvalenzuela\traspaso-innominados\data\CCLA\CONOSUR exists: True


📑 Bloque 5 – Detección y carga del archivo más reciente

Busca el archivo Excel más reciente dentro de la carpeta indicada.

Si el archivo está bloqueado (por OneDrive, etc.), se crea una copia temporal para poder leerlo.

Detecta automáticamente la hoja objetivo según los nombres configurados.

Así, el script siempre trabaja con el archivo actualizado más reciente.

In [ ]:
assert RUTA_ARCHIVOS.is_dir(), f"No existe carpeta: {RUTA_ARCHIVOS}"
archivos = [p for p in RUTA_ARCHIVOS.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
archivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
assert archivos, "No se encontraron Excel válidos."

archivo_origen = archivos[0]
print(f"Archivo más reciente: {archivo_origen.name}  |  {datetime.fromtimestamp(archivo_origen.stat().st_mtime):%Y-%m-%d %H:%M:%S}")

_tmp_copy_path = None

def _norm(s): 
    return " ".join(str(s).split()).lower()

pref_norm = {_norm(n) for n in PREFERRED_SHEETS}

try:
    try:
        with pd.ExcelFile(archivo_origen, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"Hoja objetivo: {target}")

    except PermissionError:
        _tmp_copy_path = archivo_origen.parent / f"__tmp_copy_{int(time.time()*1000)}{archivo_origen.suffix}"
        shutil.copy2(archivo_origen, _tmp_copy_path)
        print(f"No se pudo abrir el archivo original, usando copia temporal: {_tmp_copy_path.name}")

        with pd.ExcelFile(_tmp_copy_path, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"Hoja objetivo (copia): {target}")

except Exception as e:
    raise SystemExit(f"Error al abrir el Excel: {e}")

📄 Archivo más reciente: 10_2025_Facturación_Cesantía.xlsx  |  2025-12-26 13:03:00
Hojas: ['resumen', 'Cesantia_Conosur']
✅ Hoja objetivo: Cesantia_Conosur


🧽 Bloque 6 – Lectura segura y limpieza inicial

Se lee la hoja seleccionada sin encabezado para detectar filas basura.

Se eliminan las primeras filas vacías o con contenido no relevante.

Se deja el DataFrame limpio y listo para establecer los encabezados correctos.

In [ ]:
def read_excel_safe_no_header(io, sheet_name):
    try:
        return pd.read_excel(io, sheet_name=sheet_name, header=None, engine="openpyxl")
    except Exception as e:
        raise SystemExit(f"No se pudo leer la hoja '{sheet_name}': {e}")


def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip().lower()
        return s in {"", "none", "nan"}
    return False


def drop_initial_empty_rows(df: pd.DataFrame, max_check_rows=2, empty_threshold=0.8, verbose=True):
    """
    Elimina las primeras 'max_check_rows' filas si tienen >= empty_threshold de celdas vacías.
    Devuelve (df_limpio, df_preview_eliminadas)
    """
    if df.empty:
        if verbose: print("DataFrame vacío, nada que eliminar.")
        return df, pd.DataFrame()

    rows_to_drop = []
    for i in range(min(max_check_rows, len(df))):
        row = df.iloc[i]
        n_total = len(row)
        n_empty = sum(_is_nullish(x) for x in row)
        ratio = (n_empty / n_total) if n_total else 1.0
        if ratio >= empty_threshold:
            rows_to_drop.append(i)
            if verbose:
                print(f"🧹 Fila inicial {i} eliminada ({ratio:.0%} vacía).")
        else:
            if verbose:
                print(f"Fila inicial {i} conservada ({ratio:.0%} vacía).")

    removed = pd.DataFrame()
    if rows_to_drop:
        removed = df.iloc[rows_to_drop].copy()
        if verbose and not removed.empty:
            print("\nFilas eliminadas (preview):")
            try:
                display(removed)
            except Exception:
                print(removed.to_string(index=True))
        df = df.drop(index=rows_to_drop).reset_index(drop=True)
    else:
        if verbose:
            print("No se eliminaron filas iniciales.")
    return df, removed

Bloque 7 – Normalización de encabezados

Se toma la primera fila como encabezado.

Se normalizan y limpian los nombres de columnas usando las funciones del bloque anterior.

Se garantiza que todos los encabezados sean únicos para evitar conflictos.

Esto genera un DataFrame estructurado y fácil de manipular.

In [ ]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw0 = read_excel_safe_no_header(io, target)

# Elimina hasta 2 filas iniciales si están mayormente vacías
df_raw0, _removed_preview = drop_initial_empty_rows(
    df_raw0, max_check_rows=2, empty_threshold=0.8, verbose=True
)

if df_raw0.empty:
    raise SystemExit("El archivo quedó sin datos tras eliminar filas iniciales.")

def make_unique(names):
    seen = {}
    out = []
    for n in names:
        base = n if n else "col"
        if base not in seen:
            seen[base] = 1
            out.append(base)
        else:
            seen[base] += 1
            out.append(f"{base}_{seen[base]}")
    return out


header_row = df_raw0.iloc[0].astype(str).fillna("").tolist()

norm_cols = [fun.normalize_name(x) if fun.normalize_name(x) else f"col_{i+1}" for i, x in enumerate(header_row)]

norm_cols = make_unique(norm_cols)

df_raw = df_raw0.iloc[1:].reset_index(drop=True).copy()
df_raw.columns = norm_cols


dups = [c for c in df_raw.columns if df_raw.columns.tolist().count(c) > 1]
if dups:
    print("Aún hay duplicados inesperados en columnas:", sorted(set(dups)))


🧹 Fila inicial 0 eliminada (92% vacía).
🧹 Fila inicial 1 eliminada (92% vacía).

🧾 Filas eliminadas (preview):


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Tasa Prima Bruta x mil,NaN,Prima Neta Mensual,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.002,0.19,NaN,NaN


✂️ Bloque 8 – Limpieza de datos de texto

Se eliminan espacios y valores como "None", "NaN", etc.

Se estandarizan los textos para evitar errores en conversiones posteriores.

Permite mantener la consistencia en columnas de tipo texto antes del mapeo.

In [8]:
obj_cols = df_raw.select_dtypes(include=["object", "string"]).columns
if len(obj_cols) > 0:
    df_raw[obj_cols] = (
        df_raw[obj_cols]
        .apply(lambda s: s.astype(str).str.strip())
        .replace({"None": "", "none": "", "NaN": "", "nan": ""})
    )

print("Encabezados normalizados:", list(df_raw.columns))

df_raw.head()

Encabezados normalizados: ['foliocredito', 'rutafiliado', 'dvafiliado', 'nombreafiliado', 'plazo', 'montobruto', 'fecotorgamiento', 'fechaprimervto', 'fechaultimovto', 'valorcuota', 'fechaprima', 'prima', 'desgravamen', 'fechadefuncion', 'origendefuncion', 'producto', 'folioorigen', 'fechaorigen', 'tasaorigen', 'poliza', 'tasa', 'prima_bruta_mensual', 'iva', 'nan', 'nan_2']


,foliocredito,rutafiliado,dvafiliado,nombreafiliado,plazo,montobruto,fecotorgamiento,fechaprimervto,fechaultimovto,valorcuota,...,producto,folioorigen,fechaorigen,tasaorigen,poliza,tasa,prima_bruta_mensual,iva,nan,nan_2
0,171000111457,12496793,7,MORLA CASTILLO ALARCON,60,1614583,20220121,20220228,20270131,60796,...,Normal,,,0,6354562,2,3229.166,515.5811260504202,2713.58487394958,-0.166
1,174000095392,7544093,6,FERNANDO SOTO MORALES,48,1009744,20220321,20220430,20260331,45083,...,Normal,,,0,6354562,2,2019.488,322.4392605042017,1697.0487394957984,-0.488
2,106000275026,16671123,1,CARMEN SANTIBANEZ SEPULVEDA,64,2867776,20220726,20220831,20271130,105770,...,RENEGOCIACION,294000069795,20190611,2.35,6354562,2,5735.552,915.7604033613443,4819.791596638655,0.448
3,102000799383,19269109,5,Cristian Gajardo Oyarzo,60,3529234,20210218,20210331,20260228,124596,...,Normal,,,0,6354562,2,7058.468,1126.9822857142854,5931.4857142857145,-0.468
4,081001036940,16447882,3,CARMEN LEONOR ORTEGA PEDRAZA,58,1160282,20220707,20220831,20270531,51254,...,Normal,,,0,6354562,2,2320.564,370.5102184873949,1950.053781512605,0.436


Bloque 8.1 – Normalización de encabezados

Parcha para buscar las ultimas filas, y ver si son algun TOTAL o PROMEDIO.

De ser asi, se eliminan

In [9]:
def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip()
        return (s == "") or (s.lower() == "none")
    return False


def _row_nullish_ratio(row: pd.Series, exclude=()):
    cols = [c for c in row.index if c not in exclude]
    if not cols:
        return 1.0
    n_null = sum(_is_nullish(row[c]) for c in cols)
    return n_null / len(cols)


def drop_trailing_mostly_null(
    df: pd.DataFrame,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True,
):
    """
    Elimina filas al final del DF mientras tengan un ratio alto de nulos.
    Además MUESTRA las filas eliminadas con todo su contenido.
    """
    if df is None or df.empty:
        if verbose:
            print("DF vacío: nada que hacer.")
        return df

    out = df.copy()
    removed_rows = []   # 👈 aquí guardaremos (index_original, fila_entera)

    exclude = set(null_check_exclude) | set(also_exclude_money_cols)

    i = len(out) - 1
    while i >= 0:
        row = out.iloc[i]
        ratio = _row_nullish_ratio(row, exclude=exclude)

        if verbose:
            print(f"Fila índice {out.index[i]} → null_ratio={ratio:.2%}")

        if ratio >= null_ratio_threshold:
            # Guardar la fila completa ANTES de eliminarla
            removed_rows.append((out.index[i], row.copy()))

            i -= 1
        else:
            break

    # Si no hubo nada para eliminar
    if not removed_rows:
        if verbose:
            print("❎ No se detectaron filas finales mayoritariamente nulas.")
        return out

    # Mostrar filas eliminadas
    if verbose:
        print(f"\n🧹 Eliminando {len(removed_rows)} fila(s) finales mayoritariamente nulas:\n")
        for idx, row in removed_rows:
            print(f"--- Fila eliminada (índice original {idx}) ---")
            print(row.to_frame().T)   # muestra la fila completa en formato bonito
            print("\n")

    # Finalmente eliminar del DF
    drop_indices = [idx for idx, _ in removed_rows]
    out = out.drop(index=drop_indices).reset_index(drop=True)

    return out


df_raw = drop_trailing_mostly_null(
    df_raw,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True
)


Fila índice 3385 → null_ratio=92.00%
Fila índice 3384 → null_ratio=4.00%

🧹 Eliminando 1 fila(s) finales mayoritariamente nulas:

--- Fila eliminada (índice original 3385) ---
     foliocredito rutafiliado dvafiliado nombreafiliado plazo   montobruto  \
3385                                                           11479768584   

     fecotorgamiento fechaprimervto fechaultimovto          valorcuota  ...  \
3385                                                127445.45435745938  ...   

     producto folioorigen fechaorigen tasaorigen poliza tasa  \
3385                                                           

     prima_bruta_mensual iva nan nan_2  
3385                               NaN  

[1 rows x 25 columns]




🧭 Bloque 9 – Mapeado de columnas al formato estándar

Se buscan las columnas relevantes del archivo original, aunque tengan distintos nombres.

Se crea un nuevo DataFrame (df_can) con nombres normalizados y uniformes.

Este paso alinea la estructura del Excel con la esperada por la base de datos SQL.

In [10]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)

# Origen → Destino
foliocredito        = pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
rutafiliado         = pick(df_raw, "afirut", "rut_afiliado", "rut", "rutafiliado")
dvafiliado          = pick(df_raw, "afirutdv", "dv_afiliado", "dv", "dvafiliado")
NombreAfiliado      = pick(df_raw, "afinom", "nombre_afiliado", "nombre", "nombreafiliado")
Plazo               = pick(df_raw, "crecuotot", "plazo")
MontoBruto          = pick(df_raw, "cresolmon", "monto_bruto", "monto", "montobruto")
fecotorgamiento     = pick(df_raw, "fecotorgamiento", "fecha_otorgamiento", "fec_otorgamiento")
fechaPrimerVto      = pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob", "fechaprimervto")
FechaUltimoVto      = pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob", "fechaultimovto")
ValorCuota          = pick(df_raw, "crecuomon", "valor_cuota", "valorcuota")
FechaPrima          = pick(df_raw, "fechaprima", "fecha_prima")
Prima               = pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_" , "prima")
Desgravamen         = pick(df_raw, "prima_desgravamen", "desgravamen")
FechaDefuncion      = pick(df_raw, "fecdefuncion", "fecha_defuncion", "fechadefuncion")
OrigenDefuncion     = pick(df_raw, "origen_defuncion", "origin_defuncion", "origendefuncion")
Producto            = pick(df_raw, "producto")
FolioOrigen         = pick(df_raw, "folio_original", "folio_origen", "folioorigen")
FechaOrigen         = pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen", "fechaorigen")
TasaOrigen          = pick(df_raw, "tasa_interes", "tasaorigen")
Poliza              = pick(df_raw, "poliza", "póliza")
Tasa                = pick(df_raw, "tasa_interes", "tasa")
PrimaBruta          = pick(df_raw, "prima_bruta_mensual", "prima_bruta")
Iva                 = pick(df_raw, "iva")
PrimaNeta           = pick(df_raw, "primaneta", "prima_neta", "nan")


df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "NombreAfiliado": NombreAfiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fecotorgamiento": fecotorgamiento,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "FechaPrima": FechaPrima,
    "Prima": Prima,
    "Desgravamen": Desgravamen,
    "FechaDefuncion": FechaDefuncion,
    "OrigenDefuncion": OrigenDefuncion,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "TasaOrigen": TasaOrigen,
    "POLIZA": Poliza,
    "TASA": Tasa,
    "PrimaBrutaMensual": PrimaBruta,
    "IVA": Iva,
    "PrimaNetaMensual": PrimaNeta,
    "Nombre_de_archivo": archivo_origen.name,
})

df_can.head()

,foliocredito,rutafiliado,dvafiliado,NombreAfiliado,Plazo,MontoBruto,fecotorgamiento,fechaPrimerVto,FechaUltimoVto,ValorCuota,...,Producto,FolioOrigen,FechaOrigen,TasaOrigen,POLIZA,TASA,PrimaBrutaMensual,IVA,PrimaNetaMensual,Nombre_de_archivo
0,171000111457,12496793,7,MORLA CASTILLO ALARCON,60,1614583,20220121,20220228,20270131,60796,...,Normal,,,0,6354562,2,3229.166,515.5811260504202,2713.58487394958,10_2025_Facturación_Cesantía.xlsx
1,174000095392,7544093,6,FERNANDO SOTO MORALES,48,1009744,20220321,20220430,20260331,45083,...,Normal,,,0,6354562,2,2019.488,322.4392605042017,1697.0487394957984,10_2025_Facturación_Cesantía.xlsx
2,106000275026,16671123,1,CARMEN SANTIBANEZ SEPULVEDA,64,2867776,20220726,20220831,20271130,105770,...,RENEGOCIACION,294000069795,20190611,2.35,6354562,2,5735.552,915.7604033613443,4819.791596638655,10_2025_Facturación_Cesantía.xlsx
3,102000799383,19269109,5,Cristian Gajardo Oyarzo,60,3529234,20210218,20210331,20260228,124596,...,Normal,,,0,6354562,2,7058.468,1126.9822857142854,5931.4857142857145,10_2025_Facturación_Cesantía.xlsx
4,081001036940,16447882,3,CARMEN LEONOR ORTEGA PEDRAZA,58,1160282,20220707,20220831,20270531,51254,...,Normal,,,0,6354562,2,2320.564,370.5102184873949,1950.053781512605,10_2025_Facturación_Cesantía.xlsx


🗂️ Bloque 10 – Normalización del nombre del archivo

Se extrae automáticamente el período (YYYYMM) desde el nombre del archivo.

Si no se encuentra, se genera un nombre genérico con la fecha actual.

El resultado se guarda en la columna Nombre_de_archivo, clave para la trazabilidad en SQL.

In [ ]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = fun._strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'Facturacion_Cesantia_YYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"Facturacion_Cesantia_{yyyymm}.xlsx"
    except ValueError as e:
        print(f"No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"Facturacion_Cesantia_{timestamp}.xlsx"
    

nombre_canonico = canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : 10_2025_Facturación_Cesantía.xlsx
Nombre canónico : Facturacion_Cesantia_202510.xlsx


🔢 Bloque 11 – Conversión y tipificación de columnas

Se convierten columnas al tipo correcto (entero, flotante o texto).

Se limpian valores nulos, se formatean mayúsculas y se ajusta la longitud de texto.

Este paso asegura compatibilidad entre los datos procesados y la estructura SQL.

In [ ]:
INT_COLS    = ["rutafiliado", "Plazo", "fecotorgamiento","fechaPrimerVto", "FechaUltimoVto", "FechaPrima",
                "FechaDefuncion", "FechaOrigen", "POLIZA"]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["MontoBruto", "ValorCuota", "Prima", "Desgravamen", "TasaOrigen",  "TASA", "PrimaBrutaMensual", "IVA", "PrimaNetaMensual"]
STR_COLS    = ["NombreAfiliado", "OrigenDefuncion", "Producto", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"


def _to_num_series(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    return pd.to_numeric(s2, errors="coerce")

# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )

if "NombreAfiliado" in df_can.columns:
    df_can["NombreAfiliado"] = df_can["NombreAfiliado"].astype("string").str.strip().str.slice(0, 255)

if "OrigenDefuncion" in df_can.columns:
    df_can["OrigenDefuncion"] = df_can["OrigenDefuncion"].astype("string").str.strip().str.slice(0, 255)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 255)

if "Nombre_de_archivo" in df_can.columns:
    df_can["Nombre_de_archivo"] = df_can["Nombre_de_archivo"].astype("string").str.strip().str.slice(0, 50)

print("dtypes DESPUÉS:\n", df_can.dtypes)

criticas = ["foliocredito","rutafiliado","dvafiliado","Nombre_de_archivo"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "foliocredito","rutafiliado","dvafiliado","NombreAfiliado","Plazo","MontoBruto","fecotorgamiento",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","FechaPrima","Prima","Desgravamen","FechaDefuncion",
    "OrigenDefuncion","Producto","FolioOrigen", "FechaOrigen","TasaOrigen","POLIZA","TASA","PrimaBrutaMensual",
    "IVA","PrimaNetaMensual","Nombre_de_archivo"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

✅ dtypes DESPUÉS:
 foliocredito                  Int64
rutafiliado                   Int64
dvafiliado                   object
NombreAfiliado       string[python]
Plazo                         Int64
MontoBruto                  float64
fecotorgamiento               Int64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
FechaPrima                    Int64
Prima                       float64
Desgravamen                 float64
FechaDefuncion                Int64
OrigenDefuncion      string[python]
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
TasaOrigen                  float64
POLIZA                        Int64
TASA                        float64
PrimaBrutaMensual           float64
IVA                         float64
PrimaNetaMensual            float64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafili

🔍 Bloque 12 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [ ]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Facturacion_Cesantia_202510.xlsx: 3385 filas


🏁 Bloque 13 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [ ]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {data["tablas_remotas"]["conosur_acumulado_r_bkp"]}
        WHERE Nombre_de_archivo = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {data["tablas_remotas"]["conosur_acumulado_r_bkp"]}
        WHERE Nombre_de_archivo = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

        if df_sub.empty:
            print(f"No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
                  f"({existing_count} filas en {data["tablas_remotas"]["conosur_acumulado_r_bkp"]}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=data["tablas_remotas"]["conosur_acumulado_r_bkp"],
            con=conn,
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


🆕 No hay data previa para Nombre_de_archivo = 'Facturacion_Cesantia_202510.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 3385 filas para Nombre_de_archivo = 'Facturacion_Cesantia_202510.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - Facturacion_Cesantia_202510.xlsx: 3385 filas cargadas (archivo nuevo).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\nNo se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\nNo se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\nError inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\CONOSUR\09_2025_Facturación_Cesantía.xlsx


# SQL

## VALIDACIONES / INSPECCIÓN

In [ ]:
query_1 = f"""
-- [CONOSUR] Archivos cargados (desc)
    SELECT DISTINCT 
        Nombre_de_archivo
    FROM {data["tablas_remotas"]["conosur_acumulado_r_bkp"]}
    ORDER BY 
        Nombre_de_archivo desc;
"""

fun.exec_sql(query_1, connection_string)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

In [ ]:
query_2 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["conosur_final_acumulado_bkp"]};
"""

fun.exec_sql(query_2, connection_string)

In [ ]:
query_3 = f"""
SELECT *,
       /* 6354562 as POLIZA, -- comentado en tu script */
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       6832 as PLAN_TECNICO,
       4 as PLAZO_CUOTAS,
       'Credito Consumo' as Negocio
into {data["tablas_remotas"]["conosur_final_acumulado_bkp"]}
from {data["tablas_remotas"]["conosur_acumulado_r_bkp"]};
"""

fun.exec_sql(query_3, connection_string)